In [1]:
import argparse
import os
import pickle
import sys
import typing

import pandas as pd
import torch
from Bio import SeqIO
from typing import List, Union, Optional, Callable, Sequence
from transformers import (
    EsmForMaskedLM, 
    EsmConfig,
    PretrainedConfig, 
    EsmTokenizer, 
    DataCollatorForLanguageModeling, 
    Trainer
)

from tokenizers import Tokenizer
import torch
import torch.nn.functional as F
from torch import Tensor, nn

from sklearn.linear_model import LinearRegression

import einops
import yaml
import sys
import json
import functools
import os
import shutil

import numpy as np
from huggingface_hub import hf_hub_download
from peft import LoraConfig, get_peft_model
from datasets import Dataset, load_dataset
import math
from tqdm import tqdm

from matplotlib import pyplot as plt

from jaxtyping import Bool, Float, Int
from plotly.subplots import make_subplots
import plotly.express as px
from plotly_utils import (
    imshow,
    line,
    bar
)

import circuitsvis as cv
from IPython.display import display, HTML
from IPython import get_ipython
ip = get_ipython()
if not ip.extension_manager.loaded:
    ip.extension_manager.load('autoreload')
    %autoreload 2

In [2]:
import transformer_lens
import transformer_lens.utils as utils
from transformer_lens.hook_points import (
    HookedRootModule,
    HookPoint,
)

# Hooking utilities
from transformer_lens import (
    HookedTransformer,
    HookedTransformerConfig,
    FactoredMatrix,
    ActivationCache,
)

sys.path.append("../../scripts")
from compute_node_embeddings import load_sequences, get_protein_sequence
from interp_utils import get_hooked_state_dict, get_hooked_esm_config, get_logits_hooked_esm, get_fairesm_state_dict

In [3]:
from covfit_stuff.config import Config, ModelConfig
from covfit_stuff.esm_regression import load_model_for_inference, get_model_predictions, EsmForRegression
import tempfile

# Load CovFit model

In [4]:
TOK_DIR = "./covfit_stuff/Tokenizer"
CONF_DIR = "./covfit_stuff/Config"
TASK_IDS_FILE = "./covfit_stuff/task_id_dict.pt"
FOLD_ID = 0
N_TARGETS = 1565
MODEL_PATH = f"./covfit_stuff/models/covfit_model_20241007_{FOLD_ID}.ckpt"

In [5]:
# MODEL_PATH = "TheSatoLab-UTokyo/CoVFit"
# FOLD_IDS_TO_USE = [0]
# TARGET_FOLD_ID = 0
# OUTPUT_PREFIX = "inference_results"

model_name = "facebook/esm2_t33_650M_UR50D"
device = "cuda"
CONTEXT_LEN = 1024
torch.autograd.grad_mode.set_grad_enabled(False)
torch.set_float32_matmul_precision("medium")

In [6]:
def get_model(
    TOK_DIR = "./covfit_stuff/Tokenizer",
    CONF_DIR = "./covfit_stuff/Config",
    TASK_IDS_FILE = "./covfit_stuff/task_id_dict.pt",
    FOLD_ID = 0,
    N_TARGETS = 1565,
    MODEL_PATH = f"./covfit_stuff/models/covfit_model_20241007_{FOLD_ID}.ckpt",
    device=device
):
    esm_config = EsmConfig.from_pretrained(CONF_DIR)
    model = EsmForRegression(esm_config, N_TARGETS).to(device)

    lora_config = LoraConfig(
        task_type="SEQ_CLS",
        r=8,
        lora_alpha=16,
        target_modules=["key", "query", "value","dense"],
        lora_dropout=0.05,
        bias="lora_only",
        modules_to_save=["regressor"]
    )
    esm_fine_tuned = get_peft_model(model, lora_config)
    state_dict = torch.load(MODEL_PATH, map_location=device)
    
    # keys_to_remove = []
    # for key in state_dict.keys():
    #     if 'contact_head' in key:
    #         keys_to_remove.append(key)
    
    # for key in keys_to_remove:
    #     del state_dict[key]

    wrong_keys = [key for key in state_dict.keys() if key not in esm_fine_tuned.state_dict().keys()]
    key_list = list(state_dict.keys())
    for key in key_list:
        if key in wrong_keys:
            correct_key = key.rsplit('.',1)[0]+'.base_layer.'+key.rsplit('.',1)[1]
            state_dict[correct_key] = state_dict.pop(key)

    del state_dict["base_model.model.esm.embeddings.position_embeddings.base_layer.weight"]
    
    esm_fine_tuned.load_state_dict(state_dict)
    esm_fine_tuned = esm_fine_tuned.merge_and_unload()
    esm_fine_tuned.eval()
    esm_fine_tuned.esm.embeddings.token_dropout = False

    return esm_fine_tuned

In [7]:
esm_fine_tuned = get_model()
# fairesm_state_dict = get_fairesm_state_dict(esm_fine_tuned.state_dict(), esm_config)
# save_state_dict = True
# if save_state_dict:
#     torch.save(fairesm_state_dict, "./covfit_stuff/models/covfit_esm_statedict.pt")

In [8]:
esm_fine_tuned = esm_fine_tuned.to(device)
esm_fine_tuned = esm_fine_tuned.eval()

In [9]:
esm_config = esm_fine_tuned.config
esm_config.token_dropout = False
esm_config.model_name = model_name
REPO_ID = esm_config.model_name
original_task_id_infos = torch.load("./covfit_stuff/task_id_dict.pt", map_location=device)

In [10]:
tokenizer_config = {}
special_tokens_map_file = "./covfit_stuff/Tokenizer/special_tokens_map.json"
tokenizer_config["vocab_file"] = "./covfit_stuff/Tokenizer/vocab.txt"
tokenizer_config["model_max_length"] = CONTEXT_LEN

with open("./covfit_stuff/Tokenizer/special_tokens_map.json", "r") as f:
    tokenizer_config = {**tokenizer_config, **(json.load(f))}

In [11]:
with open(tokenizer_config["vocab_file"], "r") as f:
    f_data = f.read().split("\n")
    aa_to_toks_map = {i:f_data[i] for i in range(len(f_data))}
    aa_to_toks_map_rev = {aa_to_toks_map[k]:k for k in aa_to_toks_map.keys()}

In [12]:
tokenizer = EsmTokenizer(**tokenizer_config)

hooked_esm_config = get_hooked_esm_config(esm_config, context_len=CONTEXT_LEN, use_hook_tokens=True)
hooked_esm = HookedTransformer(hooked_esm_config)
print(hooked_esm.load_state_dict(get_hooked_state_dict(esm_fine_tuned.state_dict(), hooked_esm_config)))

<All keys matched successfully>


In [13]:
# clean up memory
torch.cuda.empty_cache()

# Load Data

In [14]:
def tokenizer_for_map(seq, seq_key="input_ids", tokenizer=tokenizer): #Tokenizer and params including special_tokens_mask required for MLM
    return tokenizer(
        seq[seq_key],
        return_tensors="pt", 
        return_special_tokens_mask=True,
        truncation=True,
        padding="max_length",
        max_length=300,
    )

In [15]:
# data loading
with open("../../config/pathogen_config.yaml", "r") as config_file:
    config = yaml.safe_load(config_file)
pathogens = list(config["pathogens"].keys())
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer,return_tensors='pt',mlm_probability=0.15)

In [16]:
MAX_LEN=1024
pathogen_suffixes = ["africa", "asia", "europe", "north_america", "oceania", "south_america"]
d_out_vocab = esm_fine_tuned.regressor[3].weight.size(0)
pathogen_name = "sars_cov_2_spike"
protein_coords = config["pathogens"][f"{pathogen_name}_africa"]["protein_coords"]

In [17]:
"""
uniq_seqs - seqs used in training
seq_names - names of ALL sequences
all_seqs - ALL sequences
seq_idxs - map from seq_names to uniq_seqs, i.e. seq_names[i] is for uniq_seqs[seq_idxs[i]]
"""

all_seqs = []
seq_names = []
seq_idxs = []
all_uniq_seqs = []

for suff in pathogen_suffixes:
    fasta_file = f"../../data/pathogen/{pathogen_name}_{suff}/alignment.fasta"
    data = load_sequences(fasta_file)
    sequence_names, sequences = list(zip(*list(data.items())))
    sequences = [get_protein_sequence(x, protein_coords) for x in sequences]

    keep_idx = [i for i,x in enumerate(sequences) if len(x.replace("-","")) > (CONTEXT_LEN // 5) * 4]
    sequences = [sequences[i] for i in keep_idx]
    sequence_names = [sequence_names[i] for i in keep_idx]
    
    uniq_seqs_suff, unique_inv_idx  = np.unique(sequences, return_inverse=True) # For the purpose of eval, I only care about unique sequences 

    all_seqs.extend(sequences)
    seq_names.extend(sequence_names)
    seq_idxs.extend(unique_inv_idx + len(all_uniq_seqs))
    all_uniq_seqs.extend(uniq_seqs_suff)

all_uniq_seqs, unique_inv_idx  = np.unique(all_uniq_seqs, return_inverse=True) # For the purpose of eval, I only care about unique sequences 
seq_idxs = [unique_inv_idx[idx] for idx in seq_idxs]
all_uniq_seqs = list(all_uniq_seqs)

# identical code to how it's compute_node_embeddings.py
tok_output = tokenizer(all_uniq_seqs, return_tensors="pt", return_special_tokens_mask=True, truncation=True, padding="max_length", max_length=MAX_LEN)
tok_seqs = tok_output.input_ids.to(device)
tok_masks = tok_output.attention_mask.to(device)

print(pathogen_name)
print(f"Number unique sequences: {len(all_uniq_seqs)}")
print(tok_seqs.shape)

sars_cov_2_spike
Number unique sequences: 4404
torch.Size([4404, 1024])


In [18]:
# add padding mask to model
def add_perma_hooks_to_mask_pad_tokens(
    model: HookedTransformer, pad_token: int
) -> HookedTransformer:
    # Hook which operates on the tokens, and stores a mask where tokens equal [pad]
    def cache_padding_tokens_mask(tokens: Float[Tensor, "batch seq"], hook: HookPoint) -> None:
        hook.ctx["padding_tokens_mask"] = einops.rearrange(tokens == pad_token, "b sK -> b 1 1 sK")

    # Apply masking, by referencing the mask stored in the `hook_tokens` hook context
    def apply_padding_tokens_mask(
        attn_scores: Float[Tensor, "batch head seq_Q seq_K"],
        hook: HookPoint,
    ) -> None:
        attn_scores.masked_fill_(model.hook_dict["hook_tokens"].ctx["padding_tokens_mask"], -1e5)
        if hook.layer() == model.cfg.n_layers - 1:
            del model.hook_dict["hook_tokens"].ctx["padding_tokens_mask"]

    # Add these hooks as permanent hooks (i.e. they aren't removed after functions like run_with_hooks)
    for name, hook in model.hook_dict.items():
        if name == "hook_tokens":
            hook.add_perma_hook(cache_padding_tokens_mask)
        elif name.endswith("attn_scores"):
            hook.add_perma_hook(apply_padding_tokens_mask)

    return model


hooked_esm.reset_hooks(including_permanent=True)
hooked_esm = add_perma_hooks_to_mask_pad_tokens(hooked_esm, 1)

In [19]:
component_name_map = dict()
for l in range(esm_config.num_hidden_layers + 1):
    if l < esm_config.num_hidden_layers:
        component_name_map[l] = f"blocks.{l}.hook_resid_pre"
    
    # final layer
    elif l == esm_config.num_hidden_layers:
        component_name_map[l] = f"unembed.hook_in"

In [20]:
def get_logit_hooked(output: Float[Tensor, "batch pos d_model"], tok_id):
    logits = get_logits_hooked_esm(output[:,0,:], esm_fine_tuned.regressor)[:,tok_id]
    torch.cuda.empty_cache()
    return logits

In [21]:
def get_rev_names(id_seq):
    """
    Given seq x in all_uniq_seqs, get the corresponding name(s) of sequences that have the same spike protein
    """
    if type(id_seq) == int:
        id_seq = [id_seq]

    rev_name_dict = {}
    for id_s in id_seq:
        name_idx = np.argwhere(np.array(seq_idxs) == id_s)[:,0]
        rev_name_dict[id_s] = [seq_names[x] for x in name_idx]
    return rev_name_dict

#### Given some sequence with name seq_names[i], its corresponding sequence is all_uniq_seqs[seq_idxs[i]]
#### Given some sequence with index uniq_seqs[i], its corresponding name(s) is get_rev_names(i)

In [22]:
logit_id = original_task_id_infos["fitness_USA"]

## Per-Layer Transcoder experiments (not enough GPU for Cross-layer Transcoders

## Curating annotated dataset

In [ ]:
N = 455
L455_seqs = [x for x in all_uniq_seqs if x[N-1] == "L"]
L455_idx = [i for i,x in enumerate(all_uniq_seqs) if x[N-1] == "L"]
synthetic_seqs = [x[:N-1] + "F" + x[N:] for x in L455_seqs]

In [ ]:
wt_idx = [i for i,x in enumerate(seq_names) if "wuhan" in x.lower()][0]
wt_seq = np.array(list(all_uniq_seqs[seq_idxs[wt_idx]]))[None, :]
np_seqs = np.array([list(x) for x in [*all_uniq_seqs, *synthetic_seqs]])

In [ ]:
total_cls_data_num = np_seqs.shape[0]

In [ ]:
seq_muts = [] # List[List[str]] where each element is the number of muts from element i for wt (WuHan)
num_muts = []

diffs = np.argwhere(np_seqs != wt_seq)
for s in range(np_seqs.shape[0]):
    diff_aa_idx = diffs[diffs[:,0] == s][:,1] #mutations are 1-indexed
    num_muts.append(diff_aa_idx.shape[0])
    
    seq_muts.append([f"Spike-{wt_seq[0, x]}{x+1}{np_seqs[s, x]}" if np_seqs[s, x] != "-" else f"del Spike-{wt_seq[0, x]}{x+1}" for x in diff_aa_idx])

seq_muts = [str(x)[1:-1].replace("'","") for x in seq_muts]

In [ ]:
equiv_class_seq_names = list(get_rev_names(list(range(len(all_uniq_seqs)))).values())
equiv_class_seq_names = [
    [x for x in elem if "NODE" not in x] if len([x for x in elem if "NODE" not in x]) > 0  else ["unlabeled internal node(s)"] for elem in equiv_class_seq_names 
]
equiv_class_seq_names = equiv_class_seq_names + [[f"(synthetic L455F) {x}" for x in equiv_class_seq_names[i]] for i in L455_idx]
equiv_class_seq_names = [str(x)[1:-1].replace("'","") for x in equiv_class_seq_names]

In [ ]:
for i,idx in enumerate(L455_idx):
    equiv_class_seq_names[idx] = f"(L455 p{i}) {equiv_class_seq_names[idx]}"

for i,j in enumerate(range(len(all_uniq_seqs), len(equiv_class_seq_names), 1)):
    equiv_class_seq_names[j] = f"(F455 p{i}) {equiv_class_seq_names[j]}"

In [ ]:
plt_data_path = "../../data"
clt_data_dict = pd.DataFrame.from_dict({
    "id": equiv_class_seq_names,
    "Mutations": seq_muts,
    "Num mutations": num_muts,
    "sequence": [*all_uniq_seqs, *synthetic_seqs]
})
clt_data_dict.to_parquet(os.path.join(plt_data_path, "pls_data.parquet"))

## Testing PLT

In [23]:
sys.path.append("../../ProtoMech/")
sys.path.append("../../ProtoMech/training_transcoder")
sys.path.append("../../ProtoMech/training")
sys.path.append("../../ProtoMech/function_circuit")
import argparse
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import ModelCheckpoint
from data_module import SequenceDataModule
from plt_module import PLTLightningModule
import random
from data_module import SequenceDataModule
from circuit_utils.plt_circuit import CircuitDiscovererPLT
from circuit_utils.circuit_utils import compute_attribution, rank_nodes, circuit_search
from function_utils import evaluate_circuit, evaluate_probe_direct, EmbeddingDataset
from esm.modules import ContactPredictionHead, RobertaLMHead

from scipy.stats import spearmanr

In [24]:
parser = argparse.ArgumentParser()
# Path params (defaults handled in main.sh usually, but keeping safe defaults here)
parser.add_argument("--data-dir", type=str, required=True, help="Path to .a2m or .parquet file")
parser.add_argument("--esm2-weight", type=str, required=True, help="Path to ESM2 weights .pt file")
parser.add_argument("--output-dir", type=str, default="results", help="Directory for checkpoints/logs")

# Model params
parser.add_argument("--num-layers", type=int, default=6, help="Total layers in pLM")
parser.add_argument("--d-model", type=int, default=320)
parser.add_argument("--d-hidden", type=int, default=3200, help="Latent dim per layer")

# Training params
parser.add_argument("--batch-size", type=int, default=32)
parser.add_argument("--lr", type=float, default=2e-4)
parser.add_argument("--k", type=int, default=16)
parser.add_argument("--auxk", type=int, default=32)
parser.add_argument("--dead-steps-threshold", type=int, default=10000)
parser.add_argument("--max-epochs", type=int, default=1)
parser.add_argument("--num-devices", type=int, default=1)
parser.add_argument("--wandb-project", type=str, default="ESM-CLT")

_StoreAction(option_strings=['--wandb-project'], dest='wandb_project', nargs=None, const=None, default='ESM-CLT', type=<class 'str'>, choices=None, required=False, help=None, metavar=None)

In [25]:
plt_data_path = "../../data"
arg_dict = {
    "--data-dir": os.path.join(plt_data_path, "pls_data.parquet"),
    "--esm2-weight":"./covfit_stuff/models/covfit_esm_statedict.pt",
    "--output-dir": "./covfit_stuff/PLT_test",
    "--num-layers": esm_config.num_hidden_layers,
    "--d-model": esm_config.hidden_size, # d_model of ESM
    "--d-hidden": 10 * esm_config.hidden_size, # latent dim of CLT 
    "--batch-size": 10, 
}
args = parser.parse_args([str(x) for (k,v) in arg_dict.items() for x in (k,v)])

In [26]:
run_name = f"PLT_L{args.num_layers}_H{args.d_hidden}"
run_output_dir = os.path.join("./covfit_stuff/PLT_test", run_name)
# os.makedirs(run_output_dir, exist_ok=True)

In [27]:
discoverer = CircuitDiscovererPLT(device, ckpt_path=f"{run_output_dir}/checkpoints/last.ckpt", esm_weights_path="./covfit_stuff/models/covfit_esm_statedict.pt")
discoverer.pl_module.tokenize = lambda x: tokenizer(
    x, return_tensors="pt", 
    return_special_tokens_mask=True, 
    truncation=True, 
    padding="max_length",
    max_length=CONTEXT_LEN
).input_ids.to(device)

Loading PLT from ./covfit_stuff/PLT_test/PLT_L33_H12800/checkpoints/last.ckpt...


In [28]:
del discoverer.esm.lm_head
discoverer.esm.lm_head = lambda x: x
# discoverer.esm.lm_head = RobertaLMHead(embed_dim=esm_config.hidden_size, output_dim=esm_config.vocab_size, weight=esm_fine_tuned.esm.embeddings.word_embeddings.weight)

In [29]:
data_mod = pd.read_parquet(args.data_dir)

In [30]:
N = 455 
L455_data = data_mod[["(L455" == x[:5] for x in data_mod["id"]]]
F455_data = data_mod[["(F455" == x[:5] for x in data_mod["id"]]]


CIRC_NUM_SEQS = 100
np.random.seed(1)
rand_idx = np.random.permutation(np.arange(CIRC_NUM_SEQS))
plt_circ_seqs = list(F455_data["sequence"].iloc[rand_idx])
plt_circ_seqs_fitness = esm_fine_tuned(discoverer.pl_module.tokenize(plt_circ_seqs)).logits[:,logit_id].cpu().numpy()
torch.cuda.empty_cache()

In [31]:
torch.autograd.grad_mode.set_grad_enabled(False)

torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [32]:
PLT_FLAGS = {
    "discoverer_cls": CircuitDiscovererPLT,
    # "ckpt": PLT_CHECKPOINT,
    "flags": {
        "sequential": True,         # Sequential Mode
        # "freeze_attention": False,   
        "freeze_attention": True,   # Isolate MLP/PLT
        "source": "layer_output"
    }
}
print(PLT_FLAGS)

{'discoverer_cls': <class 'circuit_utils.plt_circuit.CircuitDiscovererPLT'>, 'flags': {'sequential': True, 'freeze_attention': True, 'source': 'layer_output'}}


In [33]:
full_plt_recon_fitness = torch.empty(CIRC_NUM_SEQS).to(device)
print(full_plt_recon_fitness.shape)
for i in range(0,CIRC_NUM_SEQS,15):
    full_plt_recon_i = discoverer.reconstruct_layer_embeddings(plt_circ_seqs[i:(i+15)], source=PLT_FLAGS["flags"]["source"])
    full_plt_recon_fitness[i:(i+15)] = get_logit_hooked(full_plt_recon_i, logit_id)
    del full_plt_recon_i
    torch.cuda.empty_cache()
max_spearman = spearmanr(plt_circ_seqs_fitness, full_plt_recon_fitness.cpu().numpy())[0]
target_spearman = 0.7 * max_spearman
print(f"max spearman (full PLT): {max_spearman}\ncircuit target spearman: {target_spearman}")

torch.Size([100])
max spearman (full PLT): 0.9989918265941742
circuit target spearman: 0.6992942786159219


In [34]:
torch.autograd.grad_mode.set_grad_enabled(True)

torch.autograd.grad_mode.set_grad_enabled(mode=True)

In [35]:
# load probe (linear regression proxy of CovFit Regressor)
class cls_probe(nn.Module):
    def __init__(self, device=device):
        super().__init__()
        self.regressor = esm_fine_tuned.regressor

    def forward(self, x: Float[Tensor, "batch seq_len d_model"]):
        # only do linear regression on <cls> token
        return self.regressor(x[:,0,:])[:,logit_id]

probe = cls_probe()

In [36]:
global_attr = compute_attribution(
    discoverer, 
    probe, 
    plt_circ_seqs, 
    args.batch_size, 
    freeze_attention=PLT_FLAGS["flags"]["freeze_attention"], 
    source=PLT_FLAGS["flags"]["source"]
)

In [37]:
torch.cuda.empty_cache()

In [38]:
ranking = rank_nodes(global_attr)

In [39]:
eval_fn = lambda d, p, s, y_true, nodes, bs: evaluate_circuit(
    discoverer=d, 
    probe=p, 
    data=s, 
    y=y_true, 
    circuit_nodes=nodes, 
    batch_size=bs, 
    cnn=False, 
    **PLT_FLAGS["flags"]
)['spearman']

In [40]:
# https://github.com/amirgroup-codes/ProtoMech/blob/main/function_circuit/01_discover_circuits.py#L100
STEP_SIZE=32
MAX_NODES=1000
EXPT_NAME = "PLT"

In [41]:
best_nodes, best_k, val_recovered_spearman = circuit_search(
    discoverer=discoverer,
    probe=probe,
    ranking=ranking,
    val_seqs=plt_circ_seqs,
    val_y=plt_circ_seqs_fitness,
    target_metric=target_spearman,
    metric_fn=eval_fn,
    step_size=STEP_SIZE,
    max_nodes=MAX_NODES,
    desc=f"[{EXPT_NAME}]",
    **PLT_FLAGS["flags"]
)


[PLT]:   0%|                                                    | 0/31 [00:04<?, ?it/s, nodes=32, score=0.999, target=0.699, peak=0.999]
                                                                                                                                        

# Visualize PLT

In [42]:
sys.path.append("../../ProtoMech/steering")
from local_replacement_models import LocalCLTReplacementModel

In [43]:
N = 455
wt_idx = [i for i,x in enumerate(seq_names) if "wuhan" in x.lower()][0]
wt_seq = all_uniq_seqs[seq_idxs[wt_idx]]
wt_L455F = wt_seq[:N-1] + "F" + wt_seq[N:]

In [45]:
parser = argparse.ArgumentParser(description="Generate circuit from top activations per layer.")
parser.add_argument("--sequence", type=str, required=True, help="Protein sequence to analyze")
parser.add_argument("--output", type=str, required=True, help="Output path for circuit JSON")

_StoreAction(option_strings=['--output'], dest='output', nargs=None, const=None, default=None, type=<class 'str'>, choices=None, required=True, help='Output path for circuit JSON', metavar=None)

In [52]:
arg_dict = {
    "--sequence": wt_L455F,
    "--output":"./covfit_stuff/circuit_files/top_activations.json",
}
args = parser.parse_args([str(x) for (k,v) in arg_dict.items() for x in (k,v)])

In [53]:
TOP_K = 10  # Number of top activations per layer
replacement_model = LocalCLTReplacementModel(discoverer.pl_module, device, base_prompt=args.sequence)

# Extract top-k latent indices per layer from latents_cache
nodes = {}
for layer_idx, latents_T1D in replacement_model.latents_cache.items():
    latents_TD = latents_T1D.squeeze(1)  # (T, D)
    
    max_acts_per_latent_D, _ = latents_TD.max(dim=0)  # (D,)
    
    top_k_indices = torch.topk(max_acts_per_latent_D, k=min(TOP_K, max_acts_per_latent_D.shape[0])).indices

    # Convert to list of ints
    nodes[str(layer_idx)] = top_k_indices.cpu().tolist()
    
    print(f"Layer {layer_idx}: top {TOP_K} latents = {nodes[str(layer_idx)]}")

# Save circuit JSON
circuit = {"nodes": nodes}
with open(args.output, "w") as f:
    json.dump(circuit, f, indent=2)

print(f"Saved circuit to {args.output}")
TOP_K = 10  # Number of top activations per layer

Layer 0: top 10 latents = [2848, 12756, 9553, 8372, 6705, 4380, 61, 10230, 9135, 2318]
Layer 1: top 10 latents = [5042, 4218, 5211, 5394, 9001, 5892, 2060, 2972, 3332, 459]
Layer 2: top 10 latents = [3850, 17, 9292, 9088, 4927, 7079, 6240, 9583, 1372, 12657]
Layer 3: top 10 latents = [4389, 9228, 6556, 9129, 4770, 1377, 4994, 12269, 2317, 845]
Layer 4: top 10 latents = [12314, 12446, 9408, 9807, 5252, 4255, 2951, 5422, 10029, 11116]
Layer 5: top 10 latents = [11371, 3707, 8002, 574, 6416, 6082, 8953, 11843, 4188, 8510]
Layer 6: top 10 latents = [695, 2381, 6714, 8579, 5735, 11205, 10144, 8922, 5476, 5270]
Layer 7: top 10 latents = [1808, 9166, 3848, 9898, 4572, 2858, 2434, 7374, 4111, 3015]
Layer 8: top 10 latents = [2158, 6488, 3505, 4307, 4960, 10576, 9722, 3452, 2103, 9888]
Layer 9: top 10 latents = [12550, 1357, 4008, 10455, 2146, 12690, 2686, 8382, 1401, 5243]
Layer 10: top 10 latents = [3482, 9776, 9908, 12716, 1646, 1116, 12260, 3212, 6336, 9330]
Layer 11: top 10 latents = [5886

In [74]:
# !python circuit_analysis_function.py \
#         --sequence1 "$SEQUENCE1" \
#         --sequence2 "$SEQUENCE2" \
#         --sequence3 "$SEQUENCE3" \
#         --circuit_json "$CIRCUIT_JSON" \
#         --entry_name "$NAME"


1
